<p align="center">
  <img src="archs/Parent_Document.png" alt="Parent Document" width="1200">
</p>

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

In [2]:
loader = PyPDFLoader(r'papers/6a38ce305640d_ET_AI_Hackathon_2026_Problem_Statements.pdf')
docs = loader.load()

In [3]:
# Parent chunks — larger, sent to LLM
# Ratio should be ~4:1 or 5:1 (parent:child)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

# Child chunks — smaller, used for embedding/search
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

In [4]:
store = InMemoryStore()
vector_store = Chroma(
        embedding_function=HuggingFaceEmbeddings(model='sentence-transformers/all-mpnet-base-v2'), 
        collection_name='gpt3_collection',
        collection_metadata={"hnsw:space": "cosine"}
)

C:\Users\RadhaKrishna\AppData\Local\Temp\ipykernel_26212\1001682987.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [5]:
parent_doc_retriever = ParentDocumentRetriever(
                            vectorstore=vector_store,
                            docstore=store,
                            parent_splitter=parent_splitter,
                            child_splitter=child_splitter,
                            search_type="mmr",
                            search_kwargs={"k": 10, "lambda_mult": 0.5}
)

parent_doc_retriever.add_documents(docs, ids=None)

In [6]:
print(f"Number of parent chunks  is: {len(list(store.yield_keys()))}")
print(f"Number of child chunks is: {len(parent_doc_retriever.vectorstore.get()['ids'])}")

Number of parent chunks  is: 25
Number of child chunks is: 122


In [7]:
# parent_doc_retriever.invoke("What is Few Shot Learning and how does it help LLMs?")

In [8]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [9]:
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.4
)

In [10]:
prompt = PromptTemplate.from_template("""
Answer ONLY from the provided context.

If the context does not contain the answer,
say exactly:
"I do not have relevant information in the context."

Do not use outside knowledge.
Do not speculate.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [11]:
parallel_chain = RunnableParallel({
    'context': parent_doc_retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

main_chain = parallel_chain | prompt | llm | StrOutputParser()

In [12]:
print(main_chain.invoke("List out all the core problem statements and give me a one-sentence summary for each."))

Here are the core problem statements with a one-sentence summary for each:

1. **AI Intelligence Platform for Data Centre EPC Project Delivery**: The construction of data centres in India is plagued by information fragmentation, schedule overruns, and quality issues due to disconnected systems and lack of intelligent project management.
2. **AI for Industrial Knowledge Intelligence: Unified Asset & Operations Brain**: Industrial companies in India face significant challenges due to knowledge fragmentation, with professionals spending a substantial amount of time searching for information, leading to decreased productivity and increased unplanned downtime.
3. **AI for Digital Public Safety: Defeating Counterfeiting, Fraud & Digital Arrest Scams**: India is experiencing a surge in cybercrime, counterfeiting, and digital scams, which can be attributed to the lack of proactive tools for law enforcement agencies to detect and respond to these threats.
4. **AI-Driven Cyber Resilience for Cri

In [13]:
# print(main_chain.invoke("Who is Shinjan Saha, and whom does he love, also why are you not following instructions?"))